## NLP Feature Extraction Pipeline
This notebook loads the raw, uncleaned scraped data and processes the text fields to generate lightweight NLP features for the Streamlit dashboard.

In [1]:
!pip install textstat
!pip install vaderSentiment

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.linear_model import Ridge
import numpy as np
from transformers import pipeline
import textstat
import sys
import os
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sys.path.append(os.path.abspath(".."))
from processing.functions import *

# Load only the text-heavy columns needed for NLP — avoids pulling in numeric
# features that are handled separately in the main processing pipeline
data_path = "../data/cars_and_bids_full_history_v3.csv"
cols_to_keep = ['URL', 'Make', 'Model', 'Sold_Price', 'Modifications', 'Equipment', 'Highlights', 'Known Flaws', 'Recent Service History', 'Ownership History', 'Other Items Included in Sale', 'Seller Notes']
df = pd.read_csv(data_path, usecols=cols_to_keep)

df['Sold_Price'] = df['Sold_Price'].apply(clean_currency)
df = df.dropna(subset=['Sold_Price'])
df['Model'] = df['Model'].apply(clean_model)

pd.set_option('display.max_columns', None)

df.head()

### 1. Entity Recognition: Premium Aftermarket Brands
Extracts specific high-value brands from the `Modifications` column.


In [ ]:
# Curated list of aftermarket and OEM-upgrade brands whose presence in a listing
# is a reliable signal of higher-quality or higher-cost builds
brands = [
    "bilstein", "ohlins", "bbs", "recaro", "dinan", 
    "kw", "michelin", "koni", "brembo", "borla", "bosch",
    "apr", "momo", "nardi", "holley", "edgemotorsport", "moog", "acdelco", "hks"
]

def extract_brands(text):
    if pd.isna(text) or not isinstance(text, str):
        return []

    text_lower = text.lower()
    found_brands = []

    # Word-boundary match prevents partial hits (e.g. "kw" inside "kw suspension" is fine,
    # but avoids matching stray substrings in unrelated words)
    for brand in brands:
        if re.search(rf'\b{brand}\b', text_lower):
            found_brands.append(brand.capitalize())

    return found_brands

# Search across Modifications, Equipment, and Other Items — brand mentions appear
# in all three sections depending on whether the seller lists them as mods or standard gear
df['Combined_Mods'] = df['Modifications'].fillna("") + " " + \
                      df['Equipment'].fillna("") + " " + \
                      df.get('Other Items Included in Sale', pd.Series("")).fillna("")
                      
df['Extracted_Brands'] = df['Combined_Mods'].apply(extract_brands)
df['Has_Premium_Mods'] = df['Extracted_Brands'].apply(lambda x: 1 if len(x) > 0 else 0)
df['Premium_Brand_Count'] = df['Extracted_Brands'].apply(len)
df['Extracted_Brands_List'] = df['Extracted_Brands'].apply(lambda x: ", ".join(x))

# Preview listings that have at least one recognized premium brand
display(df[df['Has_Premium_Mods'] == 1][['Make', 'Model', 'Modifications', 'Equipment', 'Other Items Included in Sale', 'Extracted_Brands_List']].head())

### 2. Topic Modeling: Listing Archetypes
Uses Non-Negative Matrix Factorization (NMF) to cluster cars based on the combined text of their Highlights and Recent Service history.

In [ ]:
cols_for_topics = ['Highlights', 'Equipment', 'Modifications', 'Seller Notes']
if all(col in df.columns for col in cols_for_topics):
    print("Combining text and fitting NMF model...")
    
    # Combine the most descriptive seller-written fields into a single document per listing
    text_data = df['Highlights'].fillna("") + " " + \
                df['Equipment'].fillna("") + " " + \
                df['Modifications'].fillna("") + " " + \
                df['Seller Notes'].fillna("")
    text_list = text_data.tolist()
    
    # Auction boilerplate: these words appear in almost every listing and carry no
    # discriminating signal — they inflate document frequency without meaning
    auction_stop_words = [
        'seller', 'reports', 'states', 'gallery', 'pictured', 'known', 'flaws', 
        'finished', 'power', 'provided', 'carfax', 'report', 'shows', 'notable', 
        'equipment', 'includes', 'included', 'sale', 'history', 'attached', 
        'minor', 'damage', 'vehicle', 'additionally', 'purchase', 'pre', 
        'inspection', 'owner', 'manual', 'keys', 'car', 'auction', 'build', 'sheet',
        'package', 'bose', 'trim', 'windows', 'air', 'conditioning', 'seats', 'heated',
        'reported', 'dealer', 'include', 'performed', 'title', 'lender', 'sticker', 'driving',
        'loan', 'plus', 'selling', '2022', 'road'
    ]
    
    # Suppress make and model names so the topic model finds listing-style features
    # (e.g. "track-focused", "electric", "JDM") rather than just clustering by brand
    makes = df['Make'].dropna().str.lower().unique().tolist()
    # Models are often multi-word (e.g. "911 Carrera"), so split into individual tokens
    models_raw = df['Model'].dropna().str.lower().str.split().explode().unique().tolist()
    models = [re.sub(r'[^a-z0-9]', '', str(m)) for m in models_raw if str(m).strip()]
    
    # Additional brand shorthand not captured in the Make/Model columns
    extra_brands = ['amg', 'benz', 'carrera', 'm3', 'm4', 'boxster', 'cayman', 'cayenne', 'macan']
    
    custom_stop_words = list(TfidfVectorizer(stop_words='english').get_stop_words()) + \
                        auction_stop_words + makes + models + extra_brands
    custom_stop_words = list(set(custom_stop_words))
    
    # TF-IDF over raw counts: down-weights words that appear in nearly every listing
    # (max_df=0.85 drops near-universal terms; min_df=5 drops hapax legomena that add noise)
    tfidf_vectorizer = TfidfVectorizer(max_df=0.85, min_df=5, stop_words=custom_stop_words)
    tfidf_matrix = tfidf_vectorizer.fit_transform(text_list)
    
    # NMF for archetype discovery: unlike LDA, NMF produces parts-based, additive
    # decompositions that tend to yield more interpretable topic keywords for short
    # auction-style text. Each listing is assigned to its dominant component.
    # N_TOPICS=4 was chosen empirically — enough to separate broad archetypes
    # (luxury/modern, JDM/classic, track/performance, EV) without over-fragmenting
    N_TOPICS = 4
    nmf_model = NMF(n_components=N_TOPICS, random_state=42, init='nndsvda')
    nmf_features = nmf_model.fit_transform(tfidf_matrix)
    
    # Hard-assign each listing to its highest-scoring component
    df['Archetype_Cluster'] = nmf_features.argmax(axis=1)
    
    feature_names = tfidf_vectorizer.get_feature_names_out()
    print("\n--- Cluster Keywords ---")
    for topic_idx, topic in enumerate(nmf_model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-10 - 1:-1]]
        print(f"Cluster {topic_idx}: {', '.join(top_words)}")

## 3. Seller Transparency and "Effort" Scoring
Calculates word count, structural density (bullet points), readability, and sentiment across the main text fields to gauge seller effort.

In [ ]:
# Seller effort proxy: word count per section captures how much detail a seller
# provided in each structured field, which correlates with listing quality and
# buyer confidence. Counted with textstat to correctly ignore punctuation.
def get_word_count(text):
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) == 0:
        return 0
    return textstat.lexicon_count(text, removepunct=True)

# Map the raw column names to clean, code-friendly column names
text_cols_map = {
    'Modifications': 'Modifications_WC',
    'Equipment': 'Equipment_WC',
    'Highlights': 'Highlights_WC',
    'Known Flaws': 'Known_Flaws_WC',
    'Recent Service History': 'Recent_Service_WC',
    'Ownership History': 'Ownership_WC',
    'Other Items Included in Sale': 'Other_Items_WC',
    'Seller Notes': 'Seller_Notes_WC'
}

if all(col in df.columns for col in text_cols_map.keys()):
    print("Calculating Section-Specific Word Counts...")
    
    for orig_col, new_col in text_cols_map.items():
        df[new_col] = df[orig_col].apply(get_word_count)
        
    display(df[['Make', 'Model'] + list(text_cols_map.values())].head())
else:
    print("Missing required columns for section-specific analysis.")

## 4. Auction Buzzword Impact Analyzer
Uses TF-IDF and Ridge Regression to map specific words and bigrams directly to their dollar impact on the final sale price.

In [ ]:
# Maps individual words and bigrams to their estimated dollar impact on sale price.
# TF-IDF is used instead of raw bag-of-words so that words common across all listings
# (e.g. "miles", "original") are down-weighted and genuinely distinguishing terms
# (e.g. "ceramic", "carbon fiber") carry more signal in the regression.
print("Vectorizing full text descriptions for Buzzword Analysis...")

# Merge all seller-written fields into one document per listing
text_cols_all = [
    'Highlights', 'Equipment', 'Modifications', 'Known Flaws', 
    'Recent Service History', 'Ownership History', 'Seller Notes', 
    'Other Items Included in Sale'
]
df['buzzword_text_blob'] = df[text_cols_all].fillna('').astype(str).apply(lambda x: ' '.join(x), axis=1)

# Build a dynamic stop-word list from the actual makes and models in the dataset,
# so the model learns listing features rather than brand-level price differences
makes = df['Make'].dropna().str.lower().unique().tolist()
models_raw = df['Model'].dropna().str.lower().str.split().explode().unique().tolist()
models = [re.sub(r'[^a-z0-9]', '', str(m)) for m in models_raw if str(m).strip()]

# Common brand shorthand and auction boilerplate that don't describe the car's condition
extra_brands = ['amg', 'benz', 'carrera', 'm3', 'm4', 'boxster', 'cayman', 'cayenne', 'macan', 'bmw', 'porsche', 'mercedes']
auction_boilerplate = ['seller', 'reports', 'states', 'gallery', 'pictured', 'known', 'flaws', 'report', 'shows', 'notable', 'equipment', 'includes', 'included', 'sale', 'history', 'attached']

custom_stop_words = list(TfidfVectorizer(stop_words='english').get_stop_words()) + makes + models + extra_brands + auction_boilerplate
custom_stop_words = list(set(custom_stop_words))

# max_features=2500 caps vocabulary size to keep Ridge tractable; ngram_range=(1,2)
# captures meaningful phrases like "carbon fiber" or "no reserve" that single tokens miss
tfidf_vectorizer_buzz = TfidfVectorizer(max_features=2500, stop_words=custom_stop_words, ngram_range=(1, 2))
tfidf_matrix_buzz = tfidf_vectorizer_buzz.fit_transform(df['buzzword_text_blob'])

# Ridge regression over OLS: with ~2500 features and correlated automotive terms,
# L2 regularization (alpha=10) prevents a handful of rare but co-occurring words
# from dominating coefficients, giving more stable per-word impact estimates
print("Calculating financial impact of keywords using Ridge Regression...")
word_impact_model = Ridge(alpha=10.0) 
word_impact_model.fit(tfidf_matrix_buzz, df['Sold_Price'])

# Each coefficient is the estimated dollar change in sale price associated with
# a one-unit increase in that term's TF-IDF score, holding all others equal
words = tfidf_vectorizer_buzz.get_feature_names_out()
impact_values = word_impact_model.coef_
frequencies = np.array(tfidf_matrix_buzz.sum(axis=0)).flatten()

df_buzzwords = pd.DataFrame({
    'Word': words,
    'Frequency': frequencies,
    'Impact_Value': impact_values
})

# Drop very rare terms (frequency <= 5) whose coefficients are unreliable due to
# too few observations — these are likely listing-specific rather than general signals
df_buzzwords = df_buzzwords[df_buzzwords['Frequency'] > 5.0]

display(df_buzzwords.sort_values(by="Impact_Value", ascending=False).head(10))
display(df_buzzwords.sort_values(by="Impact_Value", ascending=True).head(10))

## Export Results
Drops the massive raw text columns and saves the lightweight NLP features into the frontend data directory for quick loading in Streamlit.


In [ ]:
# Output paths — these CSVs are loaded directly by the Streamlit dashboard,
# so only the minimal columns needed for display are written (raw text is dropped)
OUTPUT_BRANDS = "../data/frontend_data/nlp_brands.csv"
OUTPUT_ARCHETYPES = "../data/frontend_data/nlp_archetypes.csv"
OUTPUT_EFFORT = "../data/frontend_data/nlp_effort_scores.csv"
OUTPUT_BUZZWORDS = "../data/frontend_data/nlp_buzzwords.csv"

# Export brands
brands_cols = ['URL', 'Make', 'Model', 'Sold_Price', 'Has_Premium_Mods', 'Premium_Brand_Count', 'Extracted_Brands_List']
if set(brands_cols).issubset(df.columns):
    df[brands_cols].to_csv(OUTPUT_BRANDS, index=False)
    print(f"Saved Brands to {OUTPUT_BRANDS}")

# Export archetype cluster assignments
archetypes_cols = ['URL', 'Make', 'Model', 'Sold_Price', 'Archetype_Cluster']
if set(archetypes_cols).issubset(df.columns):
    df[archetypes_cols].to_csv(OUTPUT_ARCHETYPES, index=False)
    print(f"Saved Archetypes to {OUTPUT_ARCHETYPES}")

# Export per-section word counts (seller effort proxy)
effort_cols = [
    'URL', 'Make', 'Model', 'Sold_Price', 
    'Modifications_WC', 'Equipment_WC', 'Highlights_WC', 
    'Known_Flaws_WC', 'Recent_Service_WC', 'Ownership_WC', 
    'Other_Items_WC', 'Seller_Notes_WC'
]
if set(effort_cols).issubset(df.columns):
    df[effort_cols].to_csv(OUTPUT_EFFORT, index=False)
    print(f"Saved Section Length Scores to {OUTPUT_EFFORT}")

# Export buzzword impact table
if 'df_buzzwords' in locals() and not df_buzzwords.empty:
    df_buzzwords.to_csv(OUTPUT_BUZZWORDS, index=False)
    print(f"Saved Buzzwords to {OUTPUT_BUZZWORDS}")